# Retrain the two-stage IDS safely

Install the API dependencies with `pip install -r requirements.txt` from `ids_ensemble-api`, then select the existing `.venv-ids` kernel in VS Code.

Run cells in order using the project's `.venv-ids` Python kernel. No cell calls Render, writes to Hostinger, or changes Block/Recover. Existing model files are never overwritten.

The legacy notebook shared a fitted preprocessor between Stage A and Stage B. This notebook clones independent preprocessors, splits by attack family, removes duplicate flows before splitting, calibrates only on validation data, then evaluates the **saved and reloaded** model on untouched test data.

**Default: 14 NetFlow features** matching the current collector. Set `FEATURE_MODE = "cicids78"` for a candidate with the legacy Render feature contract. The latter requires all 78 measured features; the real MikroTik collector does not provide them all. These configurations are new experiments, not reproductions of the paper's reported metrics.

CICIDS random row splits are bootstrap checks, not proof of performance on a new network. Even deduplicated rows can share capture/session patterns. Validate against independent, labeled MikroTik captures before deployment.

In [1]:
from pathlib import Path
import os, re, json, hashlib, platform, unicodedata, time
os.environ['LOKY_MAX_CPU_COUNT'] = str(os.cpu_count() or 1)
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)
if sklearn.__version__ != '1.6.1':
    raise RuntimeError('Use scikit-learn==1.6.1 to match this API requirements.txt before training.')

Python: 3.12.10 | scikit-learn: 1.6.1


In [2]:
# Configuration: a full run may take substantial RAM and time.
SEED = 42
FEATURE_MODE = 'netflow14'  # 'netflow14' or 'cicids78'
TRAIN_MODE = 'full'        # 'smoke' is a sampled plumbing check, never a deployable model
SMOKE_ROWS_PER_CLASS = 100
TARGET_ATTACK_RECALL = 0.95
TARGET_MIN_FAMILY_RECALL = 0.80
MAX_BENIGN_FALSE_POSITIVE_RATE = 0.01

cwd = Path.cwd().resolve()
API_DIR = next((p for p in [cwd, cwd / 'ids_ensemble-api'] if (p / 'app.py').exists() and (p / 'model').is_dir()), None)
if API_DIR is None:
    raise RuntimeError('Open this notebook from the project root or ids_ensemble-api directory.')
DATASET_DIR = API_DIR.parent / 'setup' / 'Training' / 'Dataset'
OUTPUT_DIR = API_DIR / 'model' / 'candidates'

NETFLOW_FEATURES = ['destination_port', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_mean', 'bwd_packet_length_mean', 'flow_bytes_s', 'flow_packets_s', 'fwd_packets_s', 'bwd_packets_s', 'down_up_ratio', 'average_packet_size']
CICIDS_FEATURES = ['destination_port', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'down_up_ratio', 'average_packet_size', 'avg_fwd_segment_size', 'avg_bwd_segment_size', 'fwd_header_length_1', 'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate', 'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate', 'subflow_fwd_packets', 'subflow_fwd_bytes', 'subflow_bwd_packets', 'subflow_bwd_bytes', 'init_win_bytes_forward', 'init_win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'active_mean', 'active_std', 'active_max', 'active_min', 'idle_mean', 'idle_std', 'idle_max', 'idle_min']
FEATURES = {'netflow14': NETFLOW_FEATURES, 'cicids78': CICIDS_FEATURES}[FEATURE_MODE]
EXPECTED_ATTACKS = {
    'BOT', 'DDOS', 'DOS GOLDENEYE', 'DOS HULK', 'DOS SLOWHTTPTEST',
    'DOS SLOWLORIS', 'FTP-PATATOR', 'HEARTBLEED', 'INFILTRATION',
    'PORTSCAN', 'SSH-PATATOR', 'WEB ATTACK - BRUTE FORCE',
    'WEB ATTACK - SQL INJECTION', 'WEB ATTACK - XSS',
}
assert TRAIN_MODE in {'full', 'smoke'}
assert 0 < TARGET_ATTACK_RECALL <= 1
assert 0 < TARGET_MIN_FAMILY_RECALL <= 1
assert 0 <= MAX_BENIGN_FALSE_POSITIVE_RATE < 1
print('Dataset:', DATASET_DIR)
print('Mode:', TRAIN_MODE, '| Features:', len(FEATURES))

Dataset: C:\Users\perso\EJIE\Herd\Intrusion-Detection-Monitoring-System\setup\Training\Dataset
Mode: full | Features: 14


## Load, normalize, and deduplicate

All 14 attack labels must exist, with enough distinct rows for three partitions. Conflicting labels for identical selected feature vectors are excluded and reported. No identifiers or class labels enter the model. In smoke mode, sampling is per family, so the resulting class proportions are artificial and its false-positive/precision estimates are not deployment evidence.

In [3]:
def normalize_name(value):
    value = unicodedata.normalize('NFKC', str(value))
    return re.sub(r'[^a-z0-9]+', '_', value.strip().lower()).strip('_')

def normalize_label(value):
    value = unicodedata.normalize('NFKC', str(value))
    value = value.replace('\ufffd', '-').replace('\u2013', '-').replace('\u2014', '-')
    value = re.sub(r'\s+', ' ', value).strip().upper()
    value = re.sub(r'\s*-\s*', ' - ', value) if value.startswith('WEB ATTACK') else value
    return value

paths = sorted(DATASET_DIR.glob('*.csv'))
if not paths:
    raise FileNotFoundError(f'No dataset CSV files in {DATASET_DIR}')
rng = np.random.default_rng(SEED)
frames = []
for path in paths:
    print('Reading', path.name, flush=True)
    for chunk in pd.read_csv(path, chunksize=100_000, low_memory=False, encoding_errors='replace'):
        chunk.columns = [normalize_name(c) for c in chunk.columns]
        if chunk.columns.duplicated().any():
            raise ValueError(f'Duplicate normalized feature names in {path.name}')
        if 'label' not in chunk:
            raise ValueError(f'Missing label column in {path.name}')
        missing = sorted(set(FEATURES) - set(chunk.columns))
        if missing:
            raise ValueError(f'{path.name} is missing {missing}')
        selected = chunk[FEATURES].apply(pd.to_numeric, errors='coerce')
        selected = selected.replace([np.inf, -np.inf], np.nan)
        selected['label'] = chunk['label'].map(normalize_label)
        if TRAIN_MODE == 'smoke':
            selected['_key'] = rng.random(len(selected))
            frames.append(selected)
            pool = pd.concat(frames, ignore_index=True)
            frames = [pool.sort_values('_key').groupby('label', sort=False).head(SMOKE_ROWS_PER_CLASS)]
        else:
            frames.append(selected)
data = pd.concat(frames, ignore_index=True).drop(columns=['_key'], errors='ignore')
unknown = set(data['label']) - EXPECTED_ATTACKS - {'BENIGN'}
if unknown:
    raise ValueError(f'Unexpected labels: {sorted(unknown)}')
# Hash features only: remove exact duplicates AND ambiguous feature/label collisions.
data['_signature'] = pd.util.hash_pandas_object(data[FEATURES], index=False)
conflicts = data.groupby('_signature')['label'].nunique()
conflicting_signatures = conflicts[conflicts > 1].index
conflicting_rows = int(data['_signature'].isin(conflicting_signatures).sum())
original_rows = len(data)
data = data[~data['_signature'].isin(conflicting_signatures)].drop_duplicates('_signature').reset_index(drop=True)
print('Loaded rows:', original_rows, '| conflicting rows removed:', conflicting_rows, '| unique rows:', len(data))
counts = data['label'].value_counts()
if set(counts.index) != EXPECTED_ATTACKS | {'BENIGN'}:
    raise ValueError('One or more attack categories or BENIGN are missing after cleanup.')
if counts.min() < 5:
    raise ValueError('Each category needs at least five distinct, unambiguous rows; increase the sample or reconsider the feature schema.')
print(counts.to_string())

Reading ddos_dataset.pcap_ISCX.csv
Reading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Reading Friday-WorkingHours-Morning.pcap.csv
Reading Monday-WorkingHours.pcap_ISCX.csv
Reading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Reading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Reading Tuesday-WorkingHours.pcap_ISCX.csv
Reading Wednesday-workingHours.pcap_ISCX.csv
Loaded rows: 2830743 | conflicting rows removed: 62400 | unique rows: 2275239
label
BENIGN                        1860725
DOS HULK                       165640
DDOS                           127173
PORTSCAN                        90169
DOS GOLDENEYE                   10254
DOS SLOWHTTPTEST                 5164
DOS SLOWLORIS                    4856
FTP-PATATOR                      4258
SSH-PATATOR                      3068
BOT                              1880
WEB ATTACK - BRUTE FORCE         1368
WEB ATTACK - XSS                  622
INFILTRATION                       36
WEB ATTACK - SQL INJ

In [4]:
indices = np.arange(len(data))
train_idx, other_idx = train_test_split(indices, test_size=0.40, stratify=data['label'], random_state=SEED)
validation_idx, test_idx = train_test_split(other_idx, test_size=0.50, stratify=data.iloc[other_idx]['label'], random_state=SEED)
for left, right in [(train_idx, validation_idx), (train_idx, test_idx), (validation_idx, test_idx)]:
    assert not set(data.iloc[left]['_signature']) & set(data.iloc[right]['_signature'])

def partition(index):
    frame = data.iloc[index]
    return frame[FEATURES], (frame['label'] != 'BENIGN').astype(int), frame['label']

X_train, y_train, family_train = partition(train_idx)
X_validation, y_validation, family_validation = partition(validation_idx)
X_test, y_test, family_test = partition(test_idx)
for name, index in [('train', train_idx), ('validation', validation_idx), ('test', test_idx)]:
    assert set(data.iloc[index]['label']) == EXPECTED_ATTACKS | {'BENIGN'}
    print(name, len(index))
if X_train.isna().all().any():
    raise ValueError('A feature is entirely missing in training; inspect the dataset before proceeding.')

train 1365143
validation 455048
test 455048


## Fit independent pipelines

Never reuse one fitted preprocessor between these stages. The snapshot assertion below also checks that fitting Stage B does not change Stage A predictions. Stage A uses a boosted tree ensemble; Stage B uses a balanced random forest.

In [5]:
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), FEATURES)
], remainder='drop')
binary_model = Pipeline([
    ('pre', clone(preprocessor)),
    ('clf', HistGradientBoostingClassifier(max_iter=30 if TRAIN_MODE == 'smoke' else 200,
        learning_rate=0.08, max_leaf_nodes=31, class_weight='balanced', random_state=SEED)),
])
family_model = Pipeline([
    ('pre', clone(preprocessor)),
    ('clf', RandomForestClassifier(n_estimators=30 if TRAIN_MODE == 'smoke' else 200,
        max_depth=24, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)),
])
started = time.monotonic()
print('Fitting Stage A...', flush=True)
binary_model.fit(X_train, y_train)
probe = X_validation.iloc[:min(100, len(X_validation))]
probability_before_stage_b = binary_model.predict_proba(probe).copy()
print('Fitting Stage B...', flush=True)
attack_train = y_train == 1
family_model.fit(X_train.loc[attack_train], family_train.loc[attack_train])
assert binary_model.named_steps['pre'] is not family_model.named_steps['pre']
np.testing.assert_allclose(binary_model.predict_proba(probe), probability_before_stage_b, rtol=0, atol=0)
assert set(family_model.classes_) == EXPECTED_ATTACKS
attack_column = list(binary_model.classes_).index(1)
print('Training seconds:', round(time.monotonic() - started, 1))

Fitting Stage A...


c:\Users\perso\EJIE\Herd\Intrusion-Detection-Monitoring-System\.venv-ids\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\perso\EJIE\Herd\Intrusion-Detection-Monitoring-System\.venv-ids\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\perso\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\perso\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, e

Fitting Stage B...
Training seconds: 79.8


## Select the threshold using validation only

Prefer thresholds meeting all three configured validation constraints: benign false-positive rate, overall attack recall, and the worst attack family's recall. If none qualify, keep a diagnostic candidate and clearly mark it **NOT READY**. A threshold is not selected using demo outcomes or final test results.

In [6]:
validation_probability = binary_model.predict_proba(X_validation)[:, attack_column]
thresholds = np.unique(np.r_[0.0, np.quantile(validation_probability, np.linspace(0, 1, 2001)), 1.0])

def rates_at_thresholds(probabilities, labels, thresholds):
    frame = pd.DataFrame(index=thresholds)
    for label in ['BENIGN'] + sorted(EXPECTED_ATTACKS):
        scores = np.sort(probabilities[np.asarray(labels) == label])
        frame[label] = (len(scores) - np.searchsorted(scores, thresholds, side='left')) / len(scores)
    attack_rates = frame[sorted(EXPECTED_ATTACKS)]
    frame['min_family_recall'] = attack_rates.min(axis=1)
    frame['macro_attack_recall'] = attack_rates.mean(axis=1)
    attack_scores = np.sort(probabilities[np.asarray(labels) != 'BENIGN'])
    benign_scores = np.sort(probabilities[np.asarray(labels) == 'BENIGN'])
    tp = len(attack_scores) - np.searchsorted(attack_scores, thresholds, side='left')
    fp = len(benign_scores) - np.searchsorted(benign_scores, thresholds, side='left')
    frame['attack_recall'] = tp / len(attack_scores)
    frame['precision'] = np.divide(tp, tp + fp, out=np.zeros(len(tp), dtype=float), where=(tp + fp) > 0)
    return frame

curve = rates_at_thresholds(validation_probability, family_validation, thresholds)
qualifying = curve[(curve['BENIGN'] <= MAX_BENIGN_FALSE_POSITIVE_RATE)
    & (curve['attack_recall'] >= TARGET_ATTACK_RECALL)
    & (curve['min_family_recall'] >= TARGET_MIN_FAMILY_RECALL)]
validation_pass = not qualifying.empty
if validation_pass:
    ranked = qualifying.sort_values(['precision', 'min_family_recall'], ascending=False)
else:
    # Diagnostic fallback respects the FPR limit where possible; it is not approved for deployment.
    fallback = curve[curve['BENIGN'] <= MAX_BENIGN_FALSE_POSITIVE_RATE]
    ranked = (fallback if not fallback.empty else curve).sort_values(
        ['min_family_recall', 'macro_attack_recall', 'precision'], ascending=False)
threshold = float(ranked.index[0])
print('Threshold:', threshold, '| validation constraints satisfied:', validation_pass)
print(curve.loc[threshold].to_string())
if not validation_pass:
    print('NOT READY: one or more validation constraints could not be met.')

Threshold: 0.34229858575544503 | validation constraints satisfied: False
BENIGN                        0.009263
BOT                           0.704787
DDOS                          0.999607
DOS GOLDENEYE                 0.998537
DOS HULK                      0.999608
DOS SLOWHTTPTEST              0.996128
DOS SLOWLORIS                 0.995881
FTP-PATATOR                   1.000000
HEARTBLEED                    1.000000
INFILTRATION                  0.571429
PORTSCAN                      0.999778
SSH-PATATOR                   1.000000
WEB ATTACK - BRUTE FORCE      0.894161
WEB ATTACK - SQL INJECTION    0.333333
WEB ATTACK - XSS              0.895161
min_family_recall             0.333333
macro_attack_recall           0.884886
attack_recall                 0.997636
precision                     0.959990
NOT READY: one or more validation constraints could not be met.


## Save a candidate, reload, and test the complete two-stage system

Candidate paths are unique. No existing artifact is replaced. Both attack detection and final attack-type recall are reported. Stage B-only accuracy must not be presented as end-to-end accuracy. Final test failures do not trigger automatic threshold retuning.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
model_path = OUTPUT_DIR / f'ids_{FEATURE_MODE}_{TRAIN_MODE}_{stamp}.joblib'
bundle = {
    'bin': binary_model, 'mul': family_model, 'features': FEATURES,
    'best_threshold': threshold,
    'schema_version': 'mikrotik-netflow-v1' if FEATURE_MODE == 'netflow14' else 'cicids78-v1',
    'dataset_profile': 'cicids-bootstrap', 'trained_at': stamp,
    'config': {'seed': SEED, 'run_type': TRAIN_MODE, 'feature_mode': FEATURE_MODE,
        'split': 'deduplicated-family-stratified-60-20-20',
        'validation_constraints_passed': validation_pass,
        'target_attack_recall': TARGET_ATTACK_RECALL,
        'target_min_family_recall': TARGET_MIN_FAMILY_RECALL,
        'max_benign_false_positive_rate': MAX_BENIGN_FALSE_POSITIVE_RATE},
}
with model_path.open('xb') as stream:
    joblib.dump(bundle, stream)
restored = joblib.load(model_path)
assert restored['bin'].named_steps['pre'] is not restored['mul'].named_steps['pre']
np.testing.assert_allclose(restored['bin'].predict_proba(probe), binary_model.predict_proba(probe), rtol=0, atol=0)
np.testing.assert_array_equal(restored['mul'].predict(probe), family_model.predict(probe))
attack_column = list(restored['bin'].classes_).index(1)
probability = restored['bin'].predict_proba(X_test)[:, attack_column]
detected = probability >= restored['best_threshold']
predicted = np.full(len(X_test), 'BENIGN', dtype=object)
if detected.any():
    predicted[detected] = restored['mul'].predict(X_test.iloc[np.flatnonzero(detected)])
labels = ['BENIGN'] + sorted(EXPECTED_ATTACKS)
print('Binary attack detection:')
print(classification_report(y_test, detected.astype(int), labels=[0, 1], target_names=['BENIGN', 'ATTACK'], zero_division=0))
print('End-to-end classification, including Stage A misses:')
print(classification_report(family_test, predicted, labels=labels, zero_division=0))
per_family = []
truth = np.asarray(family_test)
for label in sorted(EXPECTED_ATTACKS):
    mask = truth == label
    per_family.append({'attack': label, 'support': int(mask.sum()),
        'detection_recall': float(detected[mask].mean()),
        'end_to_end_recall': float((predicted[mask] == label).mean())})
family_results = pd.DataFrame(per_family)
print(family_results.to_string(index=False))
fpr = float(detected[truth == 'BENIGN'].mean())
attack_recall = float(detected[truth != 'BENIGN'].mean())
test_pass = bool(fpr <= MAX_BENIGN_FALSE_POSITIVE_RATE and attack_recall >= TARGET_ATTACK_RECALL
    and family_results['end_to_end_recall'].min() >= TARGET_MIN_FAMILY_RECALL)
metrics = {
    'model_path': str(model_path), 'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(),
    'python_version': platform.python_version(), 'sklearn_version': sklearn.__version__,
    'run_type': TRAIN_MODE, 'feature_mode': FEATURE_MODE, 'threshold': threshold,
    'validation_constraints_passed': validation_pass, 'test_constraints_passed': test_pass,
    'bootstrap_gates_passed': bool(TRAIN_MODE == 'full' and validation_pass and test_pass),
    'production_validated': False,
    'warning': 'Independent labeled MikroTik capture validation required before deployment.',
    'benign_false_positive_rate': fpr, 'attack_recall': attack_recall,
    'per_family': per_family,
    'end_to_end_report': classification_report(family_test, predicted, labels=labels, output_dict=True, zero_division=0),
    'confusion_matrix_labels': labels,
    'confusion_matrix': confusion_matrix(family_test, predicted, labels=labels).tolist(),
}
model_path.with_suffix('.metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved candidate:', model_path)
print('Bootstrap gates passed:', metrics['bootstrap_gates_passed'], '| Production validated: False')

Binary attack detection:
              precision    recall  f1-score   support

      BENIGN       1.00      0.99      0.99    372145
      ATTACK       0.96      1.00      0.98     82903

    accuracy                           0.99    455048
   macro avg       0.98      0.99      0.99    455048
weighted avg       0.99      0.99      0.99    455048

End-to-end classification, including Stage A misses:
                            precision    recall  f1-score   support

                    BENIGN       1.00      0.99      0.99    372145
                       BOT       0.50      0.72      0.59       376
                      DDOS       1.00      1.00      1.00     25434
             DOS GOLDENEYE       0.90      1.00      0.95      2051
                  DOS HULK       0.99      1.00      0.99     33128
          DOS SLOWHTTPTEST       0.96      0.99      0.98      1033
             DOS SLOWLORIS       0.98      0.99      0.99       972
               FTP-PATATOR       0.99      1.00   

## Deployment checklist

- Investigate failed families rather than forcing selected labels or reducing the threshold until demos pass. Review data quality, ambiguity, class imbalance, schema limitations, and benign false positives.
- Neither a full run nor passing the bootstrap gates proves all real attacks can be detected. Test the frozen candidate on independent labeled MikroTik captures; do not tune repeatedly on the final test set.
- `app.py` currently loads `model/ids_ensemble.joblib`. This notebook deliberately does **not** replace that file, push to GitHub, or deploy to Render. Only promote a separately reviewed candidate after validation.
- The API should require every feature in `bundle['features']`, reject missing/non-finite values, select the probability column using the attack class, and expose a model checksum/version for diagnosis. Do not silently fill missing network measurements with zero.
- If using `netflow14`, the deployed model and collector must share the NetFlow schema. A smaller schema may not distinguish all web attack categories; validation should determine suitability.
- Rebuild demo fixtures with the frozen candidate and check **both stages**, not only Stage B. Keep evaluation separate from selecting successful examples for a demonstration.
- Retraining and deployment do not require changing Block or Recover.

## Controlled experiments: weights and feature schemas

Run after the preceding cells. These experiments use **validation data only** to select configurations and thresholds. They never evaluate new candidates on `X_test`, replace a deployed model, or call an external service. The previously inspected test set is not fresh evidence for future tuned candidates.

The 78-feature comparison uses the **same cleaned cohort and train/validation indices** as the 14-feature baseline. It keeps the first raw 78-feature row corresponding to each retained 14-feature signature. This is a controlled common-cohort comparison, not a full independent assessment of all data retained by a standalone 78-feature cleanup.

Full runs may take substantial time and RAM. Smoke mode only checks execution and cannot pass readiness. Run this section from the default `netflow14` configuration. Extra features are experimental until the production collector can measure them.

In [8]:
RUN_EXPERIMENTS = True
INCLUDE_RICH_FEATURES = True
# Maximum within-attack relative weight ratio before normalization.
FAMILY_WEIGHT_CAPS = [4.0, 12.0]
if RUN_EXPERIMENTS and FEATURE_MODE != 'netflow14':
    raise ValueError('Use FEATURE_MODE="netflow14" to construct the common cohort.')
assert all(cap >= 1 for cap in FAMILY_WEIGHT_CAPS)

In [9]:
experiment_frames = {}
if RUN_EXPERIMENTS:
    experiment_frames['netflow14'] = data[NETFLOW_FEATURES].copy()
    if INCLUDE_RICH_FEATURES:
        print('Loading 78-feature measurements for the same retained rows...', flush=True)
        kept_signatures = set(data['_signature'])
        # Match the dtypes used to hash the original concatenated 14-feature table.
        signature_dtypes = data[NETFLOW_FEATURES].dtypes.to_dict()
        rich_parts = []
        for path in paths:
            for chunk in pd.read_csv(path, chunksize=100_000, low_memory=False, encoding_errors='replace'):
                chunk.columns = [normalize_name(c) for c in chunk.columns]
                missing = sorted(set(CICIDS_FEATURES) - set(chunk.columns))
                if missing:
                    raise ValueError(f'{path.name}: missing richer inputs {missing}')
                rich = chunk[CICIDS_FEATURES].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
                signatures = pd.util.hash_pandas_object(rich[NETFLOW_FEATURES].astype(signature_dtypes), index=False)
                rich['_signature'] = signatures
                rich_parts.append(rich[rich['_signature'].isin(kept_signatures)].drop_duplicates('_signature'))
        rich = pd.concat(rich_parts, ignore_index=True).drop_duplicates('_signature').set_index('_signature')
        if not kept_signatures.issubset(set(rich.index)):
            raise RuntimeError('Could not align every retained sample with its richer measurements.')
        rich = rich.reindex(data['_signature']).reset_index(drop=True)
        np.testing.assert_allclose(rich[NETFLOW_FEATURES].to_numpy(), data[NETFLOW_FEATURES].to_numpy(), rtol=0, atol=0, equal_nan=True)
        experiment_frames['cicids78'] = rich[CICIDS_FEATURES]
        print('Aligned richer rows:', len(rich))

Loading 78-feature measurements for the same retained rows...
Aligned richer rows: 2275239


In [10]:
def training_family_weights(labels, cap):
    labels = pd.Series(np.asarray(labels))
    counts = labels[labels != 'BENIGN'].value_counts()
    reference = float(counts.median())
    factors = np.sqrt(reference / counts).clip(lower=1.0, upper=cap)
    weights = labels.map(factors).fillna(1.0).to_numpy(dtype=float)
    attack = np.asarray(labels) != 'BENIGN'
    weights[attack] /= weights[attack].mean()
    return weights


def fit_experiment(features, frame, cap):
    pre = ColumnTransformer([('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
    ]), features)], remainder='drop')
    stage_a = Pipeline([
        ('pre', clone(pre)),
        ('clf', HistGradientBoostingClassifier(max_iter=30 if TRAIN_MODE == 'smoke' else 200,
            learning_rate=0.08, max_leaf_nodes=31, class_weight='balanced', random_state=SEED)),
    ])
    stage_b = Pipeline([
        ('pre', clone(pre)),
        ('clf', RandomForestClassifier(n_estimators=30 if TRAIN_MODE == 'smoke' else 200,
            max_depth=24, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)),
    ])
    training = frame.iloc[train_idx]
    if training.isna().all().any():
        raise ValueError('An experimental training feature is entirely missing.')
    fit_options = {} if cap is None else {'clf__sample_weight': training_family_weights(family_train, cap)}
    stage_a.fit(training, y_train, **fit_options)
    probe = frame.iloc[validation_idx[:100]]
    before = stage_a.predict_proba(probe).copy()
    stage_b.fit(training.loc[np.asarray(y_train) == 1], family_train.loc[np.asarray(y_train) == 1])
    assert stage_a.named_steps['pre'] is not stage_b.named_steps['pre']
    np.testing.assert_allclose(before, stage_a.predict_proba(probe), rtol=0, atol=0)
    return stage_a, stage_b


def compare_validation(stage_a, stage_b, frame):
    validation = frame.iloc[validation_idx]
    probabilities = stage_a.predict_proba(validation)[:, list(stage_a.classes_).index(1)]
    thresholds = np.unique(np.r_[0.0, np.quantile(probabilities, np.linspace(0, 1, 2001)), 1.0])
    curve = rates_at_thresholds(probabilities, family_validation, thresholds)
    truth = np.asarray(family_validation)
    attack_positions = np.flatnonzero(truth != 'BENIGN')
    families = np.full(len(truth), 'BENIGN', dtype=object)
    families[attack_positions] = stage_b.predict(validation.iloc[attack_positions])
    recall_columns = []
    for label in sorted(EXPECTED_ATTACKS):
        mask = truth == label
        correct_scores = np.sort(probabilities[mask & (families == label)])
        column = 'correct_' + label
        curve[column] = (len(correct_scores) - np.searchsorted(correct_scores, thresholds, side='left')) / mask.sum()
        recall_columns.append(column)
    curve['min_type_recall'] = curve[recall_columns].min(axis=1)
    curve['macro_type_recall'] = curve[recall_columns].mean(axis=1)
    eligible = curve[(curve['BENIGN'] <= MAX_BENIGN_FALSE_POSITIVE_RATE)
        & (curve['attack_recall'] >= TARGET_ATTACK_RECALL)
        & (curve['min_type_recall'] >= TARGET_MIN_FAMILY_RECALL)]
    passed = not eligible.empty
    pool = eligible if passed else curve[curve['BENIGN'] <= MAX_BENIGN_FALSE_POSITIVE_RATE]
    if pool.empty:
        pool = curve
    ranked = pool.sort_values(['min_type_recall', 'macro_type_recall', 'precision'], ascending=False)
    selected = float(ranked.index[0])
    row = curve.loc[selected]
    return {
        'threshold': selected, 'validation_gates_passed': passed,
        'benign_false_positive_rate': float(row['BENIGN']),
        'attack_recall': float(row['attack_recall']),
        'min_type_recall': float(row['min_type_recall']),
        'macro_type_recall': float(row['macro_type_recall']),
        'per_family': {label: {'detection_recall': float(row[label]), 'end_to_end_recall': float(row['correct_' + label])}
            for label in sorted(EXPECTED_ATTACKS)},
    }

In [11]:
experiment_results = []
if RUN_EXPERIMENTS:
    from datetime import datetime, timezone
    import hashlib
    run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    experiment_dir = OUTPUT_DIR / ('experiments_' + run_stamp)
    experiment_dir.mkdir(parents=True, exist_ok=False)
    for schema, frame in experiment_frames.items():
        features = list(frame.columns)
        family_for_schema = None
        for cap in [None] + FAMILY_WEIGHT_CAPS:
            name = schema + ('_baseline' if cap is None else '_family_cap_' + str(cap).replace('.', '_'))
            print('Training experiment:', name, flush=True)
            # Reuse the already fitted baseline; no model is refit in place.
            if schema == 'netflow14' and cap is None:
                stage_a, stage_b = binary_model, family_model
            elif family_for_schema is None:
                stage_a, stage_b = fit_experiment(features, frame, cap)
            else:
                # Stage B training does not depend on the Stage A weight experiment.
                stage_a = clone(binary_model)
                stage_a.set_params(pre=clone(ColumnTransformer([('numeric', Pipeline([
                    ('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                ]), features)], remainder='drop')))
                stage_a.fit(frame.iloc[train_idx], y_train,
                    clf__sample_weight=training_family_weights(family_train, cap))
                stage_b = family_for_schema
            family_for_schema = stage_b
            summary = compare_validation(stage_a, stage_b, frame)
            candidate = {
                'bin': stage_a, 'mul': stage_b, 'features': features,
                'best_threshold': summary['threshold'],
                'schema_version': 'mikrotik-netflow-v1' if schema == 'netflow14' else 'cicids78-v1',
                'dataset_profile': 'cicids-bootstrap-common-netflow14-cohort', 'trained_at': run_stamp,
                'config': {'run_type': TRAIN_MODE, 'seed': SEED, 'experiment': name,
                    'family_weight_cap': cap, 'selection_partition': 'validation',
                    'independent_final_evaluation_required': True},
            }
            target = experiment_dir / (name + '.joblib')
            with target.open('xb') as stream:
                joblib.dump(candidate, stream)
            loaded = joblib.load(target)
            assert loaded['bin'].named_steps['pre'] is not loaded['mul'].named_steps['pre']
            probe = frame.iloc[validation_idx[:100]]
            np.testing.assert_allclose(loaded['bin'].predict_proba(probe), stage_a.predict_proba(probe), rtol=0, atol=0)
            np.testing.assert_array_equal(loaded['mul'].predict(probe), stage_b.predict(probe))
            summary.update(name=name, feature_mode=schema, family_weight_cap=cap,
                model_path=str(target), model_sha256=hashlib.sha256(target.read_bytes()).hexdigest(),
                run_type=TRAIN_MODE, production_validated=False,
                ready_for_deployment=False)
            target.with_suffix('.validation.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
            experiment_results.append(summary)
            print(name, '| FPR:', round(summary['benign_false_positive_rate'], 4),
                '| worst type recall:', round(summary['min_type_recall'], 4),
                '| validation gates:', summary['validation_gates_passed'], flush=True)
    comparison = pd.DataFrame([{k: v for k, v in row.items() if k != 'per_family'} for row in experiment_results])
    comparison['fpr_limit_met'] = comparison['benign_false_positive_rate'] <= MAX_BENIGN_FALSE_POSITIVE_RATE
    comparison = comparison.sort_values(['fpr_limit_met', 'validation_gates_passed', 'min_type_recall', 'macro_type_recall'], ascending=False)
    comparison.to_csv(experiment_dir / 'comparison.csv', index=False)
    (experiment_dir / 'comparison.json').write_text(json.dumps(experiment_results, indent=2), encoding='utf-8')
    print(comparison[['name', 'benign_false_positive_rate', 'attack_recall', 'min_type_recall', 'macro_type_recall', 'validation_gates_passed']].to_string(index=False))
    print('Reports saved in', experiment_dir)
    print('No automatic promotion. All candidates need fresh independent evaluation; smoke results are plumbing checks only.')

Training experiment: netflow14_baseline
netflow14_baseline | FPR: 0.0093 | worst type recall: 0.0 | validation gates: False
Training experiment: netflow14_family_cap_4_0
netflow14_family_cap_4_0 | FPR: 0.0095 | worst type recall: 0.0 | validation gates: False
Training experiment: netflow14_family_cap_12_0
netflow14_family_cap_12_0 | FPR: 0.0096 | worst type recall: 0.0 | validation gates: False
Training experiment: cicids78_baseline
cicids78_baseline | FPR: 0.0016 | worst type recall: 0.0 | validation gates: False
Training experiment: cicids78_family_cap_4_0
cicids78_family_cap_4_0 | FPR: 0.0022 | worst type recall: 0.0 | validation gates: False
Training experiment: cicids78_family_cap_12_0
cicids78_family_cap_12_0 | FPR: 0.0022 | worst type recall: 0.0 | validation gates: False
                     name  benign_false_positive_rate  attack_recall  min_type_recall  macro_type_recall  validation_gates_passed
        cicids78_baseline                    0.001631       0.999904            

### Interpret the comparison

Compare Bot detection recall AND end-to-end web attack recall at the configured benign false-positive limit. Ranking first is not a passing result. Weight caps are hyperparameters selected using validation only. Rare-class supports remain small and synthetic/duplicated samples do not provide independent evaluation evidence.

The baseline threshold in this section is recalibrated using the same end-to-end validation objective as the weighted candidates. It may differ from the earlier binary-focused baseline threshold; this keeps the experimental comparisons consistent.

A stronger 78-feature result is only an offline finding until every required feature can be collected correctly. Do not replace missing features with zeros. Keep the frozen winner separate, collect fresh labeled traffic, and evaluate it once before considering deployment. These notebook changes do not modify Render, Block, or Recover.

## Stage B experiments: fixed detection, alternative family classifiers

Run after the controlled experiments section. These comparisons keep each schema's **baseline Stage A and threshold frozen**. Only Stage B changes. Training uses attack rows from the training partition; comparison uses validation only. No test predictions, Render calls, or automatic promotion occur.

Alternatives are Extra Trees, a deeper Random Forest with more features considered per split, and multiclass histogram gradient boosting with smaller leaves. Each has its own fitted preprocessing. Class balancing is learned from training only; no duplicate or synthetic rare examples are added to validation. This does not create independent rare-class evidence.

If SQL Injection has only a few validation examples, its recall is unstable. A score of 100% on three examples is not deployment validation. More real labeled examples remain necessary.

In [12]:
RUN_STAGE_B_EXPERIMENTS = True
STAGE_B_ALTERNATIVES = ['extra_trees', 'deep_random_forest', 'hist_gradient_boosting']
# Reuse the experiment baselines from the preceding section; do not select a new Stage A here.
if RUN_STAGE_B_EXPERIMENTS and (not experiment_results or not experiment_frames):
    raise RuntimeError('Run the controlled experiments section first in the same kernel.')

In [13]:
from sklearn.ensemble import ExtraTreesClassifier


def build_family_alternative(name, features):
    if name == 'extra_trees':
        classifier = ExtraTreesClassifier(n_estimators=30 if TRAIN_MODE == 'smoke' else 200,
            max_depth=None, min_samples_leaf=1, max_features='sqrt',
            class_weight='balanced', n_jobs=-1, random_state=SEED)
    elif name == 'deep_random_forest':
        classifier = RandomForestClassifier(n_estimators=30 if TRAIN_MODE == 'smoke' else 200,
            max_depth=None, min_samples_leaf=1, max_features=0.75,
            class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
    elif name == 'hist_gradient_boosting':
        classifier = HistGradientBoostingClassifier(max_iter=20 if TRAIN_MODE == 'smoke' else 150,
            learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=5,
            l2_regularization=1.0, class_weight='balanced', random_state=SEED)
    else:
        raise ValueError('Unknown Stage B alternative: ' + name)
    pre = ColumnTransformer([('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
    ]), features)], remainder='drop')
    return Pipeline([('pre', pre), ('clf', classifier)])


def evaluate_family_validation(stage_a, stage_b, frame, threshold):
    validation = frame.iloc[validation_idx]
    truth = np.asarray(family_validation)
    probabilities = stage_a.predict_proba(validation)[:, list(stage_a.classes_).index(1)]
    detected = probabilities >= threshold
    attack_positions = np.flatnonzero(truth != 'BENIGN')
    family_predictions = stage_b.predict(validation.iloc[attack_positions])
    final_attack_predictions = np.where(detected[attack_positions], family_predictions, 'BENIGN')
    per_family = {}
    for label in sorted(EXPECTED_ATTACKS):
        mask = truth[attack_positions] == label
        per_family[label] = {
            'support': int(mask.sum()),
            'detection_recall': float(detected[attack_positions][mask].mean()),
            'stage_b_only_recall': float((family_predictions[mask] == label).mean()),
            'end_to_end_recall': float((final_attack_predictions[mask] == label).mean()),
        }
    recalls = [row['end_to_end_recall'] for row in per_family.values()]
    fpr = float(detected[truth == 'BENIGN'].mean())
    attack_recall = float(detected[attack_positions].mean())
    report = classification_report(truth[attack_positions], final_attack_predictions,
        labels=sorted(EXPECTED_ATTACKS), output_dict=True, zero_division=0)
    confusion_labels = ['BENIGN'] + sorted(EXPECTED_ATTACKS)
    return {
        'threshold': float(threshold), 'benign_false_positive_rate': fpr,
        'attack_recall': attack_recall, 'min_type_recall': float(min(recalls)),
        'macro_type_recall': float(np.mean(recalls)), 'per_family': per_family,
        'validation_gates_passed': bool(fpr <= MAX_BENIGN_FALSE_POSITIVE_RATE
            and attack_recall >= TARGET_ATTACK_RECALL and min(recalls) >= TARGET_MIN_FAMILY_RECALL),
        'attack_only_end_to_end_report': report,
        'attack_only_confusion_labels': confusion_labels,
        'attack_only_confusion_matrix': confusion_matrix(truth[attack_positions], final_attack_predictions,
            labels=confusion_labels).tolist(),
    }

In [14]:
stage_b_results = []
if RUN_STAGE_B_EXPERIMENTS:
    from datetime import datetime, timezone
    import hashlib
    stage_b_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    stage_b_dir = OUTPUT_DIR / ('stage_b_experiments_' + stage_b_stamp)
    stage_b_dir.mkdir(parents=True, exist_ok=False)
    for schema, frame in experiment_frames.items():
        baseline_summary = next(row for row in experiment_results if row['name'] == schema + '_baseline')
        baseline_path = Path(baseline_summary['model_path'])
        if hashlib.sha256(baseline_path.read_bytes()).hexdigest() != baseline_summary['model_sha256']:
            raise RuntimeError('Baseline artifact checksum mismatch.')
        baseline = joblib.load(baseline_path)
        if baseline['features'] != list(frame.columns):
            raise RuntimeError('Baseline schema and experimental data differ.')
        stage_a = baseline['bin']
        threshold = baseline['best_threshold']
        probe = frame.iloc[validation_idx[:100]]
        frozen_probabilities = stage_a.predict_proba(probe).copy()
        attack_mask = np.asarray(y_train) == 1
        for classifier_name in ['baseline'] + STAGE_B_ALTERNATIVES:
            print('Stage B:', schema, classifier_name, flush=True)
            if classifier_name == 'baseline':
                stage_b = baseline['mul']
            else:
                stage_b = build_family_alternative(classifier_name, list(frame.columns))
                stage_b.fit(frame.iloc[train_idx].loc[attack_mask], family_train.loc[attack_mask])
            assert stage_a.named_steps['pre'] is not stage_b.named_steps['pre']
            np.testing.assert_allclose(stage_a.predict_proba(probe), frozen_probabilities, rtol=0, atol=0)
            assert set(stage_b.classes_) == EXPECTED_ATTACKS
            summary = evaluate_family_validation(stage_a, stage_b, frame, threshold)
            name = schema + '_' + classifier_name
            target = stage_b_dir / (name + '.joblib')
            candidate = dict(baseline)
            candidate.update(mul=stage_b, trained_at=stage_b_stamp)
            candidate['config'] = dict(baseline.get('config', {}), stage_b_experiment=classifier_name,
                stage_a_frozen=True, selection_partition='validation', independent_final_evaluation_required=True)
            with target.open('xb') as stream:
                joblib.dump(candidate, stream)
            loaded = joblib.load(target)
            assert loaded['bin'].named_steps['pre'] is not loaded['mul'].named_steps['pre']
            np.testing.assert_allclose(loaded['bin'].predict_proba(probe), frozen_probabilities, rtol=0, atol=0)
            np.testing.assert_array_equal(loaded['mul'].predict(probe), stage_b.predict(probe))
            summary.update(name=name, feature_mode=schema, classifier=classifier_name,
                model_path=str(target), model_sha256=hashlib.sha256(target.read_bytes()).hexdigest(),
                run_type=TRAIN_MODE, selection_partition='validation', stage_a_frozen=True,
                production_validated=False, ready_for_deployment=False)
            target.with_suffix('.validation.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
            stage_b_results.append(summary)
            print('Worst type recall:', round(summary['min_type_recall'], 4),
                '| macro type recall:', round(summary['macro_type_recall'], 4),
                '| validation gates:', summary['validation_gates_passed'], flush=True)
    columns = ['name', 'feature_mode', 'classifier', 'run_type', 'threshold',
        'benign_false_positive_rate', 'attack_recall', 'min_type_recall', 'macro_type_recall', 'validation_gates_passed']
    stage_b_comparison = pd.DataFrame([{key: row[key] for key in columns} for row in stage_b_results])
    stage_b_comparison = stage_b_comparison.sort_values(['validation_gates_passed', 'min_type_recall', 'macro_type_recall'], ascending=False)
    stage_b_comparison.to_csv(stage_b_dir / 'comparison.csv', index=False)
    (stage_b_dir / 'comparison.json').write_text(json.dumps(stage_b_results, indent=2), encoding='utf-8')
    print(stage_b_comparison.to_string(index=False))
    print('Saved:', stage_b_dir)
    print('Validation ranking only. No fresh final evaluation or automatic promotion has occurred.')

Stage B: netflow14 baseline
Worst type recall: 0.0 | macro type recall: 0.7992 | validation gates: False
Stage B: netflow14 extra_trees
Worst type recall: 0.0 | macro type recall: 0.7992 | validation gates: False
Stage B: netflow14 deep_random_forest
Worst type recall: 0.0 | macro type recall: 0.7989 | validation gates: False
Stage B: netflow14 hist_gradient_boosting
Worst type recall: 0.0 | macro type recall: 0.8156 | validation gates: False
Stage B: cicids78 baseline
Worst type recall: 0.0 | macro type recall: 0.8313 | validation gates: False
Stage B: cicids78 extra_trees
Worst type recall: 0.3145 | macro type recall: 0.8549 | validation gates: False
Stage B: cicids78 deep_random_forest
Worst type recall: 0.0 | macro type recall: 0.8307 | validation gates: False
Stage B: cicids78 hist_gradient_boosting
Worst type recall: 0.5693 | macro type recall: 0.8878 | validation gates: False
                            name feature_mode             classifier run_type  threshold  benign_false_p

### Stage B results and next decision

Compare the per-family validation supports, confusion matrices, and end-to-end recall. A stronger Stage B score cannot recover flows rejected by frozen Stage A. A passing validation row still requires a frozen configuration and fresh independent labeled traffic evaluation. If no classifier passes, report the limitations and collect richer or additional labeled data; do not relabel samples or tune on the previously inspected test set.

Alternative trees and smaller leaves can overfit rare examples. Class balancing changes the fitting objective, not the quantity of independent evidence. More training on the same few SQL Injection flows is not a substitute for new captures.